In [1]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from datetime import datetime
import pydicom
from pathlib import Path
import matplotlib.pyplot as plt
import os
import napari
import numpy as np
import cv2
import SimpleITK as sitk
import json
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

In [2]:
# --- Variables globales para almacenar archivos ---
archivos_ct = []  # Lista de cortes CT
archivo_spect = ""  # Archivo único SPECT

def seleccionar_fichero():
    global archivo_spect
    ruta = filedialog.askopenfilename(   # Abre un cuadro de diálogo para seleccionar un archivo de imagen SPECT.
        title="Seleccionar fichero SPECT",
        filetypes=[("Archivos DICOM", "*.dcm")]
    )
    if ruta:
        archivo_spect = ruta
        entry_spect.delete(0, tk.END) # Limpia cualquier ruta de archivo que estuviera allí antes.
        entry_spect.insert(0, ruta) # Inserta el contenido de la variable ruta
        mostrar_imagen_dicom(ruta, frame_spect, "SPECT")


def seleccionar_ct():
    global archivos_ct
    carpeta = filedialog.askdirectory(title="Seleccionar carpeta con cortes CT (.dcm)")
    if carpeta:
        archivos_ct = sorted([os.path.join(carpeta, f) for f in os.listdir(carpeta) if f.lower().endswith(".dcm")])
        if archivos_ct:
            entry_ct.delete(0, tk.END)
            entry_ct.insert(0, carpeta)
            
            # Mostrar corte central
            corte_central = archivos_ct[len(archivos_ct)//2]
            mostrar_imagen_dicom(corte_central, frame_ct)
        else:
            messagebox.showwarning("Atención", "No se encontraron archivos .dcm en la carpeta seleccionada.")
    
def leer_dicom(ruta):
    """Lee un archivo DICOM y devuelve información básica."""
    try:
        ds = pydicom.dcmread(ruta) # (DICOM Read) abre el archivo, lo analiza y carga todos sus datos (metadatos e imagen) en una variable (ds).
        info = [
            f"Nombre del paciente: {getattr(ds, 'PatientName', 'Desconocido')}",
            f"ID del paciente: {getattr(ds, 'PatientID', 'N/A')}",
            f"Modalidad: {getattr(ds, 'Modality', 'N/A')}",
            f"Dimensiones: {ds.Rows} x {ds.Columns}",
        ]
        return "\n".join(info)
    except Exception as e:
        messagebox.showerror("Error al leer DICOM", f"No se pudo leer el archivo:\n{e}")
        return None

def guardar_datos():
    """Valida y muestra los datos introducidos y DICOMs."""
    fecha = entry_fecha.get()
    actividad = entry_actividad.get()

    if not archivo_spect or not archivos_ct or not fecha or not actividad:
        messagebox.showwarning("Campos incompletos", "Por favor, complete todos los campos y seleccione SPECT y CT.")
        return

    # Validar formato de fecha
    try:
        datetime.strptime(fecha, "%d/%m/%Y")
    except ValueError:
        messagebox.showerror("Error de formato", "La fecha debe tener el formato dd/mm/aaaa.")
        return

    # Leer información del SPECT
    info_spect = leer_dicom(archivo_spect)
    if not info_spect:
        return

    # Leer información del primer corte CT como ejemplo
    info_ct = leer_dicom(archivos_ct[0])
    if not info_ct:
        return

    resumen = (
        f"Datos introducidos correctamente\n\n"
        f"Archivo SPECT: {archivo_spect}\n"
        f"Carpeta CT: {entry_ct.get()} ({len(archivos_ct)} cortes)\n"
        f"Fecha: {fecha}\n"
        f"Actividad: {actividad} MBq\n\n"
        f"Información del SPECT:\n{info_spect}\n\n"
        f"Información del primer corte CT:\n{info_ct}"
    )
    messagebox.showinfo("Resumen", resumen)

def mostrar_imagen_dicom(ruta, frame, titulo=""):
    ds = pydicom.dcmread(ruta)
    img = ds.pixel_array.astype(float)

    # SPECT multiframe
    if img.ndim == 3:
        img = img[img.shape[0] // 2]  # corte central

    # Corrección CT
    if getattr(ds, "Modality", "") == "CT":
        slope = getattr(ds, "RescaleSlope", 1)
        intercept = getattr(ds, "RescaleIntercept", 0)
        img = img * slope + intercept

    # Normalización segura
    vmin, vmax = img.min(), img.max()
    if vmax > vmin:
        img = (img - vmin) / (vmax - vmin)
    else:
        img = np.zeros_like(img)

    for widget in frame.winfo_children():
        widget.destroy()

    # Ajustamos la figura para que sea cuadrada o proporcional
    fig, ax = plt.subplots(figsize=(5, 5)) 
    
    # CAMBIO CLAVE: Usar aspect="equal" para mantener proporciones
    ax.imshow(img, cmap="gray", aspect="equal") 
    
    ax.set_title(titulo)
    ax.axis("off")
    
    # Ajusta los márgenes automáticamente para eliminar espacios vacíos
    fig.tight_layout()

    canvas = FigureCanvasTkAgg(fig, master=frame)
    canvas.draw()
    canvas.get_tk_widget().pack(fill="both", expand=True)

    plt.close(fig)



import tkinter as tk
from tkinter import ttk


# CONFIGURACIÓN DE LA VENTANA PRINCIPAL

root = tk.Tk()
root.title("Ingreso de datos SPECT")
root.geometry("1300x700")
root.resizable(False, False)

# ESTILOS

style = ttk.Style()
style.configure("TLabel", font=("Calibri", 11))
style.configure("TButton", font=("Calibri", 10))
style.configure("TEntry", font=("Calibri", 10))

# FRAMES PRINCIPALES

frame_controles = ttk.Frame(root)
frame_controles.grid(row=0, column=0, sticky="ns", padx=10, pady=10)

frame_spect_container = ttk.LabelFrame(root, text="Imagen SPECT")
frame_spect_container.grid(row=0, column=1, sticky="nsew", padx=10, pady=10)

frame_ct_container = ttk.LabelFrame(root, text="Imagen CT")
frame_ct_container.grid(row=0, column=2, sticky="nsew", padx=10, pady=10)

# Configuración del grid principal
root.columnconfigure(0, weight=0)  # Controles
root.columnconfigure(1, weight=1)  # SPECT
root.columnconfigure(2, weight=1)  # CT
root.rowconfigure(0, weight=1)


# FRAMES INTERNOS PARA IMÁGENES
frame_spect = ttk.Frame(frame_spect_container)
frame_spect.pack(fill="both", expand=True)

frame_ct = ttk.Frame(frame_ct_container)
frame_ct.pack(fill="both", expand=True)

frame_spect_container.config(width=500, height=500)
frame_ct_container.config(width=500, height=500)
frame_spect_container.grid_propagate(False)
frame_ct_container.grid_propagate(False)


# CONTROLES DE ENTRADA DE DATOS

# --- Fichero SPECT ---
ttk.Label(frame_controles, text="Fichero SPECT:").grid(row=0, column=0, sticky="w", pady=4)
entry_spect = ttk.Entry(frame_controles, width=35)
entry_spect.grid(row=0, column=1, pady=4)

ttk.Button(frame_controles, text="Examinar...", command=seleccionar_fichero).grid(row=0, column=2, padx=5, pady=4)

# --- Carpeta CT ---
ttk.Label(frame_controles, text="Carpeta CT:").grid(row=1, column=0, sticky="w", pady=4)
entry_ct = ttk.Entry(frame_controles, width=35)
entry_ct.grid(row=1, column=1, pady=4)

ttk.Button(frame_controles, text="Examinar...", command=seleccionar_ct).grid(row=1, column=2, padx=5, pady=4)

# --- Fecha de adquisición ---
ttk.Label(frame_controles, text="Fecha de adquisición (dd/mm/aaaa):").grid(row=2, column=0, sticky="w", pady=4)
entry_fecha = ttk.Entry(frame_controles, width=20)
entry_fecha.grid(row=2, column=1, pady=4, sticky="w")

# --- Actividad ---
ttk.Label(frame_controles, text="Actividad (MBq):").grid(row=3, column=0, sticky="w", pady=4)
entry_actividad = ttk.Entry(frame_controles, width=20)
entry_actividad.grid(row=3, column=1, pady=4, sticky="w")

# BOTONES DE ACCIÓN
ttk.Button(frame_controles, text="Guardar datos", command=guardar_datos).grid(row=4, column=1, pady=12, sticky="w")

# EJECUCIÓN DE LA APLICACIÓN
root.mainloop()


In [ ]:
def load_dicom_folder(path):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(path)
    reader.SetFileNames(dicom_names)
    image = reader.Execute()
    return image

In [9]:
ct_path = r"C:\Users\gervi\OneDrive - Universidad Complutense de Madrid (UCM)\MASTER\SEGUNDO CUATRI\TFM\Imagenes prueba\CT"

ct = load_dicom_folder(ct_path)
print(ct.GetSize())
print(ct.GetSpacing())
print(ct.GetOrigin())

(512, 512, 606)
(0.976562, 0.976562, 1.25)
(-250.0, -250.0, -378.35)


VISOR NAPARI PARA SPECT

In [6]:
# --- Cargar el DICOM ---
file_path = r"C:\Users\gervi\OneDrive - Universidad Complutense de Madrid (UCM)\MASTER\SEGUNDO CUATRI\TFM\Imagenes prueba\Tomo4FOV_Lu177peak_IRACSC001_DS.dcm"
ds = pydicom.dcmread(file_path)
image = ds.pixel_array.astype(float)

# --- Lanzar visor ---
viewer = napari.view_image(image, name="Lu-177 Tomografía", colormap='gray', contrast_limits=[0, np.max(image)])
napari.run()

C:\Users\gervi\AppData\Local\Temp\ipykernel_30204\1818084049.py:7: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(image, name="Lu-177 Tomografía", colormap='gray', contrast_limits=[0, np.max(image)])


VISOR NAPARI PARA CT

In [7]:
# --- Carpeta con los DICOM ---
folder = r"C:\Users\gervi\OneDrive - Universidad Complutense de Madrid (UCM)\MASTER\SEGUNDO CUATRI\TFM\Imagenes prueba\CT"

# --- Leer todos los archivos .dcm y ordenarlos (opcional: por nombre o por SliceLocation) ---
dicom_files = [f for f in os.listdir(folder) if f.endswith(".dcm")]
dicom_files.sort()  # si los nombres siguen el orden de los cortes

# --- Cargar los DICOM y extraer arrays ---
slices = []
for f in dicom_files:
    ds = pydicom.dcmread(os.path.join(folder, f))
    slices.append(ds.pixel_array.astype(float))

# --- Apilar en un array 3D (Z, Y, X) ---
image_3d = np.stack(slices, axis=0)

# --- Lanzar visor 3D en Napari ---
viewer = napari.view_image(
    image_3d, 
    name="Lu-177 Tomografía", 
    colormap='gray', 
    contrast_limits=[0, np.max(image_3d)]
)
napari.run()

C:\Users\gervi\AppData\Local\Temp\ipykernel_30204\2111650818.py:18: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(
